In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Remove the ID column
df_raw = pd.read_csv('./data/cirrhosis.csv').drop(columns=["ID", "N_Days"])

target_col = 'Status'

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]

In [ ]:
from sklearn.preprocessing import LabelEncoder

y = LabelEncoder().fit_transform(y)

### Manually splitting the dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

print(f'Train lenght: {len(X_train)}')
print(f'Val lenght: {len(X_val)}')
print(f'Test lenght: {len(X_val)}')

#### Pipelines

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
num_cols = list(X.select_dtypes(include=['number']).columns)
cat_cols = list(X.select_dtypes(exclude=['number']).columns)

print(f'Num: {num_cols}')
print(f'Cat: {cat_cols}')

##### Num 

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline

num_si_mean_only_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean'))
])

num_knn_only_pipeline = Pipeline([
    ('knn imputer', KNNImputer())
])

num_ss_si_mean_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean')),
    ('standard scaler', StandardScaler())
])

num_mm_si_mean_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean')),
    ('minmax scaler', MinMaxScaler())
])

num_ss_knn_pipeline = Pipeline([
    ('knn imputer', KNNImputer()),
    ('standard scaler', StandardScaler())
])

num_mm_knn_pipeline = Pipeline([
    ('knn imputer', KNNImputer()),
    ('minmax scaler', MinMaxScaler())
])

##### Cat

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_si_unspec_ohe_pipeline = Pipeline([
    ('Unpecified simple imputer', SimpleImputer(strategy='constant', fill_value='Unspecified')),
    ('one hot encoder', OneHotEncoder(handle_unknown='ignore'))
])

#### Training & Evaluation

In [ ]:
num_pipelines = {
    'Mean Simple Imputer (No Scaling)': num_si_mean_only_pipeline,
    'Mean Simple Imputer + Standard Scaler': num_ss_si_mean_pipeline,
    'Mean Simple Imputer + MinMax Scaler': num_mm_si_mean_pipeline,
    'KNN Imputer (No Scaling)': num_knn_only_pipeline,
    'KNN Imputer + Standard Scaler': num_ss_knn_pipeline,
    'KNN Imputer + MinMax Scaler': num_mm_knn_pipeline
}

cat_pipelines = {
    'Constant Imputer (Unspecified) + OneHotEncoder': cat_si_unspec_ohe_pipeline
}

from prepare_models import create_default_models_dict

models = create_default_models_dict()

In [ ]:
from utils import create_evaluation_dataframe

results_df = create_evaluation_dataframe(
    X_train,
    y_train,
    X_val,
    y_val,
    num_pipelines,
    cat_pipelines,
    models
)

results_df

In [ ]:
# We find rows without missing categorical data
mask_clean = X[cat_cols].notna().all(axis=1)

X_clean = X[mask_clean]
y_clean = y[mask_clean]

# New train/val/test split on cleaned data
X_train_cl_tmp, X_test_clean, y_train_cl_tmp, y_test_clean = train_test_split(X_clean, y_clean, test_size=0.15, random_state=42)
X_train_clean, X_val_clean, y_train_clean, y_val_clean = train_test_split(X_train_cl_tmp, y_train_cl_tmp, test_size=0.15/0.85, random_state=42)

print(f"X_train_clean: {X_train_clean.shape}")
print(f"X_val_clean: {X_val_clean.shape}")
print(f"X_test_clean: {X_test_clean.shape}")

In [ ]:
# Pipeline for cleaned categorical data
from sklearn.preprocessing import OneHotEncoder

cat_pipelines_clean = {
    'Dropped missing Cat + OneHotEncoder': Pipeline([
        ('one hot encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
}

results_df_clean = create_evaluation_dataframe(
    X_train_clean,
    y_train_clean,
    X_val_clean,
    y_val_clean,
    num_pipelines,
    cat_pipelines_clean,
    models
)

results_df_clean

In [ ]:
df_imputed = results_df.copy()
df_imputed['method'] = 'categorical_imputed'

df_dropped = results_df_clean.copy()
df_dropped['method'] = 'categorical_dropped'

combined_results = pd.concat([df_imputed, df_dropped], ignore_index=True)
display(combined_results)

In [ ]:
a = ['RandomForest', 'SVC', 'DecisionTree', 'GaussianNB']

for model in a:
    if model in combined_results['model'].values:
        display(combined_results[combined_results['model'] == model].sort_values('val_accuracy', ascending=False))
        print()